In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import math
import os

# ==========================================
# 1. Loss Definitions (Geometrically Tuned for 2D)
# ==========================================

class DistArcLoss(nn.Module):
    def __init__(self, in_features, out_features, m=0.1, lam=0.0005, radial_gap=0.5):
        super(DistArcLoss, self).__init__()
        self.m = m
        self.lam = lam
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        
        self.W = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.kaiming_uniform_(self.W)

        radii = np.arange(2.0, 2.0 + out_features * radial_gap, radial_gap).astype('float32')
        self.register_buffer('r', torch.tensor(radii))

    def forward(self, x, labels):
        batch_size = x.size(0)
        x_norm = F.normalize(x, p=2, dim=1)
        W_norm = F.normalize(self.W, p=2, dim=0)
        
        cosine = torch.matmul(x_norm, W_norm).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        target_logit = cosine[torch.arange(batch_size), labels]
        
        sine = torch.sqrt((1.0 - torch.pow(target_logit, 2)).clamp(min=1e-7))
        marginal_cosine = target_logit * self.cos_m - sine * self.sin_m 

        W_scaled = W_norm * self.r
        target_w_r = W_scaled[:, labels].T 
        R = -target_w_r + x 
        
        R_norm = F.normalize(R, p=2, dim=1)
        target_w_r_norm = F.normalize(-target_w_r, p=2, dim=1)
        cos_phi = torch.sum(R_norm * target_w_r_norm, dim=1) 

        x_sq = torch.sum(x**2, dim=1, keepdim=True) 
        w_r_sq = torch.sum(W_scaled**2, dim=0, keepdim=True) 
        dist_sq = x_sq + w_r_sq - 2 * torch.matmul(x, W_scaled) 

        num_exp = marginal_cosine + cos_phi - (self.lam * dist_sq[torch.arange(batch_size), labels])
        denom_terms = cosine - (self.lam * dist_sq)
        denom_terms[torch.arange(batch_size), labels] = marginal_cosine
        
        loss = -num_exp + torch.logsumexp(denom_terms, dim=1)
        return loss.mean()

    def predict(self, x):
        W_scaled = F.normalize(self.W, p=2, dim=0) * self.r
        x_sq = torch.sum(x**2, dim=1, keepdim=True)
        w_r_sq = torch.sum(W_scaled**2, dim=0, keepdim=True)
        dist_sq = x_sq + w_r_sq - 2 * torch.matmul(x, W_scaled)
        return torch.argmin(dist_sq, dim=1)


class DecoupledLoss(nn.Module):
    # m and s are updated via init parameters in the run loop
    def __init__(self, in_features, out_features, s=10.0, m=0.1, r_min=2.0, lambda_=0.1, gamma=0.01):
        super(DecoupledLoss, self).__init__()
        self.s = s
        self.m = m
        self.r_min = r_min
        self.lambda_ = lambda_
        self.gamma = gamma

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

        self.W = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.W)
        self.a = nn.Parameter(torch.zeros(out_features))

    def forward(self, x, labels):
        B = x.size(0)
        r_x = torch.sqrt(torch.sum(x**2, dim=1, keepdim=True) + 1e-8)
        x_hat = x / r_x
        W_norm = F.normalize(self.W, p=2, dim=1)

        cos_theta = F.linear(x_hat, W_norm).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        target_cos = cos_theta[torch.arange(B), labels]
        
        sine = torch.sqrt((1.0 - target_cos**2).clamp(min=1e-7))
        cos_theta_m = target_cos * self.cos_m - sine * self.sin_m
        cos_theta_m = torch.where(target_cos > self.th, cos_theta_m, target_cos - self.mm)

        logits = cos_theta.clone()
        logits[torch.arange(B), labels] = cos_theta_m
        logits = logits * self.s

        L_ang = F.cross_entropy(logits, labels)

        r_all = self.r_min + F.softplus(self.a)
        r_y = r_all[labels]
        L_rad = F.mse_loss(r_x.squeeze(), r_y)
        L_reg = torch.var(r_all, unbiased=False) 

        loss = L_ang + self.lambda_ * L_rad + self.gamma * L_reg
        return loss

    def predict(self, x):
        x_hat = F.normalize(x, p=2, dim=1)
        W_norm = F.normalize(self.W, p=2, dim=1)
        return torch.argmax(F.linear(x_hat, W_norm), dim=1)


class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, s=10.0, m=0.1):
        super(ArcFaceLoss, self).__init__()
        self.s = s
        self.m = m
        self.W = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.W)
        
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, x, labels):
        W_norm = F.normalize(self.W, p=2, dim=1)
        x_norm = F.normalize(x, p=2, dim=1)
        
        cos_theta = F.linear(x_norm, W_norm).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        target_cos = cos_theta[torch.arange(x.size(0)), labels]
        
        sine = torch.sqrt((1.0 - target_cos**2).clamp(min=1e-7))
        cos_theta_m = target_cos * self.cos_m - sine * self.sin_m
        cos_theta_m = torch.where(target_cos > self.th, cos_theta_m, target_cos - self.mm)

        logits = cos_theta.clone()
        logits[torch.arange(x.size(0)), labels] = cos_theta_m
        logits = logits * self.s

        return F.cross_entropy(logits, labels)

    def predict(self, x):
        x_norm = F.normalize(x, p=2, dim=1)
        W_norm = F.normalize(self.W, p=2, dim=1)
        return torch.argmax(F.linear(x_norm, W_norm), dim=1)


class CrossEntropyLossWrapper(nn.Module):
    def __init__(self, in_features, out_features):
        super(CrossEntropyLossWrapper, self).__init__()
        self.fc = nn.Linear(in_features, out_features)

    def forward(self, x, labels):
        logits = self.fc(x)
        return F.cross_entropy(logits, labels)

    def predict(self, x):
        return torch.argmax(self.fc(x), dim=1)


# ==========================================
# 2. Model Architecture (Lightweight ResNet for CIFAR)
# ==========================================

class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class CIFARNet(nn.Module):
    def __init__(self, embed_size=2):
        super(CIFARNet, self).__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.fc = nn.Linear(256 * 4 * 4, embed_size)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, 2)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out


# ==========================================
# 3. Training & Evaluation Pipeline
# ==========================================

def plot_features(features, labels, num_classes, title, filename):
    colors = plt.cm.tab10(np.linspace(0, 1, num_classes))
    plt.figure(figsize=(10, 10))
    
    for i in range(num_classes):
        idx = labels == i
        plt.scatter(features[idx, 0], features[idx, 1], c=[colors[i]], label=str(i), alpha=0.5, s=5)
        
    plt.title(title)
    plt.legend(loc='upper right', markerscale=3)
    plt.grid(True, linestyle='--', alpha=0.3)
    
    max_radius = np.max(np.linalg.norm(features, axis=1)) if len(features) > 0 else 10
    circles = [plt.Circle((0, 0), r, color='gray', fill=False, linestyle='--', alpha=0.2) 
               for r in np.linspace(2, max_radius, 6)]
    ax = plt.gca()
    for circle in circles:
        ax.add_patch(circle)
        
    plt.xlim(-max_radius*1.1, max_radius*1.1)
    plt.ylim(-max_radius*1.1, max_radius*1.1)
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Saved visualization to {filename}")

def run_experiment(loss_type='decoupled', embed_size=2, epochs=20, batch_size=128):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n--- Running Pipeline: {loss_type.upper()} | Embed Size: {embed_size} | Dataset: CIFAR-10 ---")

    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    dataset_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    dataset_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    
    train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size, shuffle=False, num_workers=2)

    model = CIFARNet(embed_size=embed_size).to(device)
    num_classes = 10
    
    # --- The Geometric Constraints ---
    # Here we enforce the strict 2D space hyperparameters
    if loss_type == 'distarc':
        criterion = DistArcLoss(in_features=embed_size, out_features=num_classes, m=0.1, lam=0.0005).to(device)
    elif loss_type == 'decoupled':
        criterion = DecoupledLoss(in_features=embed_size, out_features=num_classes, s=10.0, m=0.1).to(device)
    elif loss_type == 'arcface':
        criterion = ArcFaceLoss(in_features=embed_size, out_features=num_classes, s=10.0, m=0.1).to(device)
    elif loss_type == 'crossentropy':
        criterion = CrossEntropyLossWrapper(in_features=embed_size, out_features=num_classes).to(device)

    # Lowered LR for stability
    optimizer = optim.SGD([
        {'params': model.parameters()},
        {'params': criterion.parameters()}
    ], lr=0.005, momentum=0.9, weight_decay=5e-4)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(1, epochs + 1):
        
        # DISTARC LAMBDA WARMUP
        if loss_type == 'distarc':
            # Softly increase the radial penalty every 4 epochs
            if epoch % 4 == 0 and criterion.lam < 0.005:
                criterion.lam += 0.001
                print(f"   [Warmup] DistArc Lambda increased to: {criterion.lam:.4f}")

        model.train()
        criterion.train()
        running_loss = 0.0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            
            embeddings = model(data)
            loss = criterion(embeddings, target)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0) 
            optimizer.step()
            running_loss += loss.item()
            
        scheduler.step()
        print(f"Epoch {epoch}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

    model.eval()
    criterion.eval()
    correct = 0
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            embeddings = model(data)
            
            preds = criterion.predict(embeddings)
            correct += preds.eq(target).sum().item()
            
            if embed_size == 2:
                all_features.append(embeddings.cpu().numpy())
                all_labels.append(target.cpu().numpy())

    accuracy = 100. * correct / len(test_loader.dataset)
    print(f"Test Accuracy for {loss_type.upper()} (Dim={embed_size}): {accuracy:.2f}%")

    if embed_size == 2:
        features = np.vstack(all_features)
        labels = np.concatenate(all_labels)
        plot_title = f"{loss_type.capitalize()} Loss CIFAR-10 Latent Space (2D)"
        plot_filename = f"cifar10_{loss_type}_2d_visualization.png"
        plot_features(features, labels, num_classes, plot_title, plot_filename)
        
    return accuracy


if __name__ == "__main__":
    epochs = 100
    
    # We only run 2D for now as requested
    acc_arc = run_experiment(loss_type='arcface', embed_size=2, epochs=epochs)
    acc_ce = run_experiment(loss_type='crossentropy', embed_size=2, epochs=epochs)
    acc_distarc = run_experiment(loss_type='distarc', embed_size=2, epochs=epochs)
    acc_decoupled = run_experiment(loss_type='decoupled', embed_size=2, epochs=epochs)
    
    print("\n================ CIFAR-10 2D SUMMARY ================")
    print(f"ArcFace Accuracy:       {acc_arc:.2f}%")
    print(f"Cross Entropy Accuracy: {acc_ce:.2f}%")
    print(f"DistArc Accuracy:       {acc_distarc:.2f}%")
    print(f"Decoupled Accuracy:     {acc_decoupled:.2f}%")


--- Running Pipeline: ARCFACE | Embed Size: 2 | Dataset: CIFAR-10 ---


100%|██████████| 170M/170M [00:10<00:00, 15.7MB/s]


Epoch 1/100 - Loss: 2.5552
Epoch 2/100 - Loss: 1.9444
Epoch 3/100 - Loss: 1.8388
Epoch 4/100 - Loss: 1.7611
Epoch 5/100 - Loss: 1.7086
Epoch 6/100 - Loss: 1.6348
Epoch 7/100 - Loss: 1.5865
Epoch 8/100 - Loss: 1.5294
Epoch 9/100 - Loss: 1.4800
Epoch 10/100 - Loss: 1.4502
Epoch 11/100 - Loss: 1.4086
Epoch 12/100 - Loss: 1.3696
Epoch 13/100 - Loss: 1.3539
Epoch 14/100 - Loss: 1.3288
Epoch 15/100 - Loss: 1.3037
Epoch 16/100 - Loss: 1.2597
Epoch 17/100 - Loss: 1.2397
Epoch 18/100 - Loss: 1.2107
Epoch 19/100 - Loss: 1.1898
Epoch 20/100 - Loss: 1.1615
Epoch 21/100 - Loss: 1.1357
Epoch 22/100 - Loss: 1.1141
Epoch 23/100 - Loss: 1.0950
Epoch 24/100 - Loss: 1.0832
Epoch 25/100 - Loss: 1.0524
Epoch 26/100 - Loss: 1.0299
Epoch 27/100 - Loss: 0.9883
Epoch 28/100 - Loss: 0.9876
Epoch 29/100 - Loss: 0.9649
Epoch 30/100 - Loss: 0.9526
Epoch 31/100 - Loss: 0.9207
Epoch 32/100 - Loss: 0.9043
Epoch 33/100 - Loss: 0.8780
Epoch 34/100 - Loss: 0.8595
Epoch 35/100 - Loss: 0.8387
Epoch 36/100 - Loss: 0.8221
E